# Potato Sorting - Stage 2 Classification
Dieses Notebook lädt den bestehenden Detection-Datensatz aus Roboflow, schneidet die Kartoffeln anhand der Bounding-Boxen aus (`Crop`) und sortiert sie automatisch in die Ordner `healthy` und `bad`, je nachdem ob sie sich mit einer `bad`- oder `cut`-Box überschneiden. Anschließend wird ein YOLO11-cls Modell darauf trainiert.

In [ ]:
%pip install -q roboflow ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace("ms-workspace-m1gci").project("my-first-project-h62tp")
dataset = project.version(8).download("yolov8")

print(f"Datensatz erfolgreich geladen: {dataset.location}")

In [ ]:
import os
import cv2
import yaml
import shutil
from pathlib import Path

# Pfade definieren
dataset_path = Path(dataset.location)
out_dir = Path("/content/potato-cls")

# Alten Ordner löschen falls vorhanden
if out_dir.exists():
    shutil.rmtree(out_dir)

# data.yaml lesen um Klassen-IDs zu bekommen
with open(dataset_path / "data.yaml", 'r') as f:
    data_yaml = yaml.safe_load(f)
    
names = data_yaml.get('names', [])
if isinstance(names, dict):
    class_names = {v: k for k, v in names.items()}
else:
    class_names = {name: i for i, name in enumerate(names)}

potato_id = class_names.get('potato', 2)
bad_id = class_names.get('bad', 0)
cut_id = class_names.get('cut', 1)
defect_ids = [bad_id, cut_id]

def check_intersection(box1, box2):
    # box: [x_center, y_center, width, height] in relativen Koordinaten
    b1_x1 = box1[0] - box1[2]/2
    b1_x2 = box1[0] + box1[2]/2
    b1_y1 = box1[1] - box1[3]/2
    b1_y2 = box1[1] + box1[3]/2
    
    b2_x1 = box2[0] - box2[2]/2
    b2_x2 = box2[0] + box2[2]/2
    b2_y1 = box2[1] - box2[3]/2
    b2_y2 = box2[1] + box2[3]/2
    
    # Überschneidung testen
    if b1_x1 > b2_x2 or b2_x1 > b1_x2: return False
    if b1_y1 > b2_y2 or b2_y1 > b1_y2: return False
    return True

healthy_count = 0
bad_count = 0

for split in ['train', 'valid', 'test']:
    split_dir = dataset_path / split
    if not split_dir.exists(): continue
        
    (out_dir / split / 'healthy').mkdir(parents=True, exist_ok=True)
    (out_dir / split / 'bad').mkdir(parents=True, exist_ok=True)
    
    for img_path in (split_dir / 'images').glob("*.jpg"):
        label_path = split_dir / 'labels' / (img_path.stem + ".txt")
        if not label_path.exists(): continue
            
        img = cv2.imread(str(img_path))
        if img is None: continue
        h, w, _ = img.shape
        
        with open(label_path, 'r') as f:
            lines = f.readlines()
            
        boxes = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cls_id = int(parts[0])
            box = [float(x) for x in parts[1:5]]
            boxes.append((cls_id, box))
            
        potato_boxes = [b for b in boxes if b[0] == potato_id]
        defect_boxes = [b for b in boxes if b[0] in defect_ids]
        
        for i, (_, p_box) in enumerate(potato_boxes):
            is_bad = False
            for _, d_box in defect_boxes:
                if check_intersection(p_box, d_box):
                    is_bad = True
                    break
                    
            # Ausschneiden (Crop)
            px_c, py_c, pw, ph = p_box
            x1 = max(0, int((px_c - pw/2) * w))
            y1 = max(0, int((py_c - ph/2) * h))
            x2 = min(w, int((px_c + pw/2) * w))
            y2 = min(h, int((py_c + ph/2) * h))
            
            crop = img[y1:y2, x1:x2]
            if crop.size == 0: continue
                
            if is_bad:
                label = 'bad'
                bad_count += 1
            else:
                label = 'healthy'
                healthy_count += 1
                
            save_path = out_dir / split / label / f"{img_path.stem}_crop_{i}.jpg"
            cv2.imwrite(str(save_path), crop)

print(f"Klassifikations-Datensatz erfolgreich erstellt in {out_dir}")
print(f"Gesunde Kartoffeln (healthy): {healthy_count}")
print(f"Kranke Kartoffeln (bad): {bad_count}")

In [ ]:
from ultralytics import YOLO

# YOLO11s Klassifikations-Modell laden
model = YOLO('yolo11s-cls.pt')

# Modell auf dem neu erstellten Datensatz trainieren
# imgsz=224 ist Standard für Klassifikation, kann man auch auf 128 oder 256 setzen
results = model.train(data='/content/potato-cls', epochs=30, imgsz=224, batch=32, plots=True)

In [ ]:
# Ergebnisse anzeigen (Confusion Matrix etc.)
from IPython.display import Image, display
import glob

for img_path in glob.glob('runs/classify/train/*.png'):
    display(Image(filename=img_path))

print("Training abgeschlossen. Checke die Confusion Matrix für die Genauigkeit von 'bad' vs 'healthy'.")